#  Lesson 3: Feed-Forward, Layer Norm & Residuals

### __The parts that make a Transformer actually trainable__

# The Problem First

Multi-head attention (Lesson 2) is fantastic at one thing: letting every token gather relevant information from every other token. But two practical problems remain if you just stack attention layers directly on top of each other:

__Problem 1__ — Attention barely "thinks." For a fixed set of attention weights, the output at each position is just a weighted linear combination of other tokens' value vectors. That's powerful for gathering the right information, but it doesn't give the model much room to nonlinearly process what it gathered. Stack ten linear operations and, mathematically, they can collapse toward something not much richer than one.

__Problem 2__ — Deep stacks don't train. GPT-3 has 96 layers. If you tried stacking 96 attention layers directly, training would fail almost immediately — the exact vanishing-gradient disease we diagnosed with RNNs back in Stage 2, Lesson 4, except now the gradient has to survive backpropagating through layers instead of through time.

This lesson fixes both. The feed-forward network gives every token real nonlinear processing power. LayerNorm + residual connections are what let you stack dozens or hundreds of layers and still train reliably.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

class feedforward(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.1):
        super().__init__()
        self.Linear1=nn.Linear(d_model,d_ff)
        self.Linear2=nn.Linear(d_ff,d_model)
        self.dropout=nn.Dropout(dropout)

    def forward(self,x):
        x=self.Linear1(x)
        x=F.gelu(x)
        x=self.dropout(x)
        x=self.Linear2(x)
        return x

d_model,d_ff=512,2048
ffn=feedforward(d_model,d_ff)
x=torch.randn(2,10,d_model)
out=ffn(x)
print("Input: ", x.shape)
print("Output:", out.shape, "— same shape, every token individually transformed")




Input:  torch.Size([2, 10, 512])
Output: torch.Size([2, 10, 512]) — same shape, every token individually transformed


__Why expand to 4x and then contract back? Projecting into a higher-dimensional space gives the nonlinearity more "room" to carve out complex functions before compressing back down to d_model. 4x is what the original paper found worked well, and it stuck — though modern models use slightly different ratios (more on that below).__

# Part B : Layer Normalization 

Deep networks are fragile: as data flows through many layers, activation values can drift, balloon, or collapse, forcing you to use tiny learning rates just to avoid blowing up. LayerNorm re-centers and re-scales every token's vector at every layer, so the network sees a consistent, stable distribution no matter how deep it is.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.gamma=nn.Parameter(torch.ones(d_model))
        self.beta=nn.Parameter(torch.zeros(d_model))
        self.eps=eps

    def forward(self,x):
        mean=x.mean(dim=1,keepdim=True)
        var=x.var(dim=1,keepdim=True,unbiased=False)
        x_norm=(x-mean)/torch.sqrt(var+self.eps)
        return self.gamma*x_norm+self.beta
    
ln = LayerNorm(d_model=8)
x  = torch.randn(2, 5, 8) * 10 + 50    # deliberately huge, shifted values
out = ln(x)
print("Before — mean:", x.mean(dim=-1)[0,0].item(), " std:", x.std(dim=-1)[0,0].item())
print("After  — mean:", out.mean(dim=-1)[0,0].item(), " std:", out.std(dim=-1)[0,0].item())
print("\n→ Renormalized to mean≈0, std≈1 — no matter how large the raw values were.")

AttributeError: module 'torch.nn' has no attribute 'Parameters'